# GuideLLM Benchmark Results Analysis

This notebook only **reads and summarises** an existing `results.json` — it does not run the benchmark itself.

### Prerequisites

1. **You must have run the `guidellm` benchmark first.** Follow the `guidellm` section of the [README](./README.md#guidellm) in a **terminal**, with the vLLM server running. That command writes a `results.json` file into this directory, which the cell below loads.
2. **Select the `.venv` kernel** (top-right of the toolbar). If the cell fails with `ModuleNotFoundError`, the wrong kernel is selected.

If the next cell fails with `FileNotFoundError: results.json`, the benchmark has not been run yet (or was run from a different directory) — go back and run the `guidellm` command first.

In [17]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the guidellm benchmark results
with open('results.json', 'r') as f:
    results = json.load(f)

# Extract benchmark data
benchmarks = results['benchmarks']

# Create a summary dataframe
summary_data = []
for i, benchmark in enumerate(benchmarks):
    metrics = benchmark['metrics']
    strategy = benchmark['config']['strategy']['type_']
    
    # Get rate if it's a constant strategy
    rate = benchmark['config']['strategy'].get('rate', None)
    
    summary_data.append({
        'Run': i + 1,
        'Strategy': strategy,
        'Rate': rate,
        'Mean Latency (s)': metrics['request_latency']['successful']['mean'],
        'Median Latency (s)': metrics['request_latency']['successful']['median'],
        'P99 Latency (s)': metrics['request_latency']['successful']['percentiles']['p99'],
        'Requests/Second': metrics['requests_per_second']['successful']['mean'],
        'Total Requests': metrics['request_totals']['successful'],
        'Duration (s)': benchmark['duration']
    })

df = pd.DataFrame(summary_data)

# Display summary statistics
print("=" * 80)
print("GUIDELLM BENCHMARK SUMMARY")
print("=" * 80)
print(f"\nTotal Benchmark Runs: {len(benchmarks)}")
print(f"\nBenchmark Results:")
print(df.to_string(index=False))




GUIDELLM BENCHMARK SUMMARY

Total Benchmark Runs: 10

Benchmark Results:
 Run    Strategy      Rate  Mean Latency (s)  Median Latency (s)  P99 Latency (s)  Requests/Second  Total Requests  Duration (s)
   1 synchronous       NaN          0.297399            0.290825         0.577812         3.359372              94     27.981417
   2  throughput       NaN          6.200474            6.831711         9.029911        10.399577              94      9.038829
   3    constant  4.239398          0.328654            0.327678         0.400833         4.221043              94     22.269378
   4    constant  5.119423          0.341600            0.332051         0.585692         5.081402              94     18.498833
   5    constant  5.999449          0.353626            0.344351         0.551517         5.934567              94     15.839405
   6    constant  6.879475          0.381170            0.373507         0.670659         6.752028              94     13.921744
   7    constant  7.7595

### Reading the table

Each row is one benchmark run at a different load:

- The **synchronous** row sends one request at a time — its latency is the unavoidable single-request floor (network + preprocessing + inference + postprocessing).
- The **throughput** row removes the rate cap to find the server's absolute ceiling; latency here is high because requests are queueing.
- The **constant** rows sweep between those two extremes at a fixed requests/second each.

What you are looking for is the **knee**: the rate at which P99 latency stops being flat and starts climbing sharply. That rate is the practical capacity of this server for this workload — the number you would use for capacity planning. (See the worked example in the [README](./README.md#analyzing-guidellm-results).)